# Generic Android firmware → TWRP Device Tree

**Upload and run. No filenames need to be typed.** Choose the original boot/recovery image mode or the modern Full firmware importer (ZIP/TAR/TAR.MD5, nested factory packages and supported partition images). Both modes produce an unverified source candidate.


In [ ]:
#@title 1. Set up or update TDTNext
from pathlib import Path
import os, shutil, subprocess

repo = Path('/content/twrpdtgen-NEXT')
url = 'https://github.com/RectifyData/twrpdtgen-NEXT.git'
if (repo / '.git').is_dir():
    subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only'], check=True)
elif repo.exists():
    raise RuntimeError(f'{repo} exists but is not a Git checkout. Move it aside and rerun.')
else:
    subprocess.run(['git', 'clone', url, str(repo)], check=True)
os.chdir(repo)

packages = {'cpio':'cpio', 'lz4':'lz4', 'xz':'xz-utils', 'bzip2':'bzip2',
            'gzip':'gzip', 'unzip':'unzip', 'zip':'zip', 'file':'file'}
missing = [package_name for command, package_name in packages.items() if not shutil.which(command)]
if missing:
    subprocess.run(['apt-get', 'update'], check=True)
    subprocess.run(['apt-get', 'install', '-y', '--no-install-recommends', *missing], check=True)

subprocess.run(['sh', str(repo / 'tdtnext'), '--setup-only'], check=True)
next_python = str(repo / '.tdtnext-venv/bin/python')
commit = subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip()
tool_version = subprocess.check_output(
    [next_python, '-c', 'import twrpdtgen; print(twrpdtgen.__version__)'],
    text=True).strip()
print(f'TDTNext {tool_version} @ {commit} ready')


## Upload firmware or images

Select stock images or archives. For very large files, upload using the Colab Files sidebar into a fresh folder and enter its path in existing_folder instead.


In [ ]:
#@title 2. Select files
from pathlib import Path
import os, tempfile
from google.colab import files

existing_folder = '' #@param {type:'string'}
if existing_folder.strip():
    input_dir = Path(existing_folder).expanduser().resolve()
    if not input_dir.is_dir():
        raise ValueError('The supplied folder does not exist.')
else:
    input_dir = Path(tempfile.mkdtemp(prefix='tdtnext-input-', dir='/content'))
    previous_cwd = Path.cwd()
    try:
        os.chdir(input_dir)
        uploaded = files.upload()
        if not uploaded:
            raise RuntimeError('No files selected. Rerun this cell to upload.')
        for name, data in uploaded.items():
            target = input_dir / Path(name).name
            if not target.is_file():
                target.write_bytes(data)
        del uploaded
    finally:
        os.chdir(previous_cwd)
print('Input folder:', input_dir)
for path in sorted(input_dir.iterdir()):
    print(' ', path.name)


## Optional settings

Boot/recovery images is the original lightweight workflow. Full firmware mode uses the current importer with vendor/odm/super analysis when available. It excludes stock-root from the ZIP. Allow partial firmware only when you accept extraction blockers recorded in the audit.


In [ ]:
#@title 3. Generation settings
input_mode = 'Boot/recovery images' #@param ['Boot/recovery images', 'Full firmware']
skeleton = 'auto' #@param ['auto', 'omni', 'twrp-12.1', 'twrp-14.1']
recovery_layout = 'auto' #@param ['auto', 'dedicated', 'boot', 'vendor_boot', 'init_boot']
recovery_root_mode = 'auto' #@param ['auto', 'minimal', 'full']
recovery_fragment = '' #@param {type:'string'}
include_stock_root_in_zip = False #@param {type:'boolean'}
allow_partial_firmware = False #@param {type:'boolean'}


In [ ]:
#@title 4. Generate and download
from pathlib import Path, PurePosixPath
import shutil, subprocess, tempfile, zipfile
from google.colab import files

workspace = Path(tempfile.mkdtemp(prefix='tdtnext-output-', dir='/content'))
tree_output = workspace / 'tree'
common = ['--skeleton', skeleton, '--recovery-layout', recovery_layout,
          '--recovery-root-mode', recovery_root_mode]
if recovery_fragment.strip():
    common += ['--recovery-fragment', recovery_fragment.strip()]

if input_mode == 'Full firmware':
    if include_stock_root_in_zip:
        raise ValueError('Full firmware archives exclude stock-root. Turn off include_stock_root_in_zip.')
    archive = tree_output / 'twrp-device-tree.zip'
    args = [next_python, '-m', 'twrpdtgen', '--cli', '--input', str(input_dir),
            '--output', str(tree_output), *common]
    if allow_partial_firmware:
        args.append('--allow-partial')
elif input_mode == 'Boot/recovery images':
    if allow_partial_firmware:
        raise ValueError('allow_partial_firmware applies only to Full firmware mode.')
    images = workspace / 'images'
    images.mkdir()
    def copy_image(source, name):
        out = images / name
        if out.exists():
            raise RuntimeError(f'Duplicate image filename: {name}')
        with out.open('wb') as dst:
            shutil.copyfileobj(source, dst, 1024 * 1024)
    for path in sorted(input_dir.iterdir()):
        if not path.is_file():
            continue
        if path.suffix.lower() == '.img':
            with path.open('rb') as src:
                copy_image(src, path.name)
        elif path.suffix.lower() == '.zip':
            if not zipfile.is_zipfile(path):
                raise ValueError(f'Invalid ZIP: {path.name}')
            with zipfile.ZipFile(path) as bundle:
                for info in bundle.infolist():
                    pure = PurePosixPath(info.filename)
                    if (info.is_dir() or pure.is_absolute() or '..' in pure.parts
                            or ((info.external_attr >> 16) & 0o170000) == 0o120000
                            or pure.suffix.lower() != '.img'):
                        continue
                    with bundle.open(info) as src:
                        copy_image(src, pure.name)
    if not any(images.iterdir()):
        raise RuntimeError('No .img files found. For TAR/MD5/nested firmware choose Full firmware.')
    archive = workspace / 'twrp-device-tree.zip'
    args = [next_python, '-m', 'twrpdtgen.auto_tree', str(images),
            '-o', str(tree_output), '--archive', str(archive), *common]
    if include_stock_root_in_zip:
        args.append('--include-stock-root')
else:
    raise ValueError(f'Unknown input mode: {input_mode}')

subprocess.run(args, check=True)
if not archive.is_file() or not zipfile.is_zipfile(archive):
    raise RuntimeError(f'No valid source ZIP was generated: {archive}')
print(f'Ready: {archive} ({archive.stat().st_size} bytes)')
files.download(str(archive))


## Results

Both modes download a source ZIP automatically. Each run uses a fresh output directory. Full firmware mode also includes firmware-input-audit.json. Review generation-report.json, partition geometry, AVB, encryption and hardware support before building or flashing.
